<a href="https://colab.research.google.com/github/LaurenMitchell-tech/uvvisml/blob/main/uvvisml_demo_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    try:
        import chemprop
    except ImportError:
        !git clone https://github.com/chemprop/chemprop.git
        %cd chemprop
        !pip install .

import pandas as pd
import numpy as np
import torch
from lightning import pytorch as pl
from pathlib import Path

from chemprop import data, featurizers, models

In [ ]:
os.chdir('/content')
!git clone https://github.com/learningmatter-mit/uvvisml
os.chdir('/content/uvvisml/uvvisml')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!tar -xvzf "/content/drive/MyDrive/School/models.tar.gz" -C "/content/uvvisml/uvvisml"

## Change Test Set Input and Load

In [ ]:
test_path = 'data/splits/lambda_max_abs/deep4chem/group_by_smiles/smiles_target_test.csv'
df_test = pd.read_csv(test_path)
df_test

## Functions

In [ ]:
def make_prediction(model_list, test_data_loader):
    all_preds_list = []

    for model in model_list:
        with torch.inference_mode():
            trainer = pl.Trainer(
                logger=None,
                enable_progress_bar=True,
                accelerator="cpu",
                devices=1
            )
            test_preds = trainer.predict(model, test_data_loader)

        test_preds = np.concatenate(test_preds, axis=0).ravel()
        all_preds_list.append(test_preds)
    return all_preds_list

## Make Predictions

##Predict experimental peak with model trained on combined training set

In [ ]:
#Change model input and load model
checkpoint_paths = ['models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_0/model.pt',
                   'models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_1/model.pt',
                   'models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_2/model.pt',
                   'models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_3/model.pt',
                   'models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_4/model.pt']
models_list = []
for path in checkpoint_paths:
  model = models.multi.MulticomponentMPNN.load_from_checkpoint(path, map_location="cpu", weights_only=False)
  models_list.append(model)

#Get Smiles
smiles_columns = ["smiles","solvent"]
smiss = df_test[smiles_columns].values
print(smiss[:5])

#Get molecule datapoints
n_components = len(smiles_columns)
test_datapointss = [[data.MoleculeDatapoint.from_smi(smi) for smi in smiss[:, i]] for i in range(n_components)]

#Get molecule dataset
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=featurizers.MultiHotAtomFeaturizer.v1())
test_dsets = [data.MoleculeDataset(test_datapoints, featurizer) for test_datapoints in test_datapointss]
test_mcdset = data.MulticomponentDataset(test_dsets)
test_loader = data.build_dataloader(test_mcdset, shuffle=False)


all_preds = make_prediction(models_list, test_loader)

all_preds_array = np.stack(all_preds, axis=1)
columns = [f"pred_{i+1}" for i in range(len(models_list))]
df_test[columns] = all_preds_array
df_test['pred_avg'] = np.mean(all_preds_array, axis=1)
df_test

## Predict TDDFT peak in vacuum

In [ ]:
#Change model input and load model
checkpoint_paths = ['models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_0/model.pt',
                    'models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_1/model.pt',
                    'models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_2/model.pt',
                    'models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_3/model.pt',
                    'models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_4/model.pt']

models_list = []
for path in checkpoint_paths:
  model = models.MPNN.load_from_checkpoint(path, map_location="cpu")
  models_list.append(model)

#Get Smiles
smiles_columns = "smiles"
smis = df_test[smiles_columns]
print(smis)

#Get molecule datapoints
test_data = [data.MoleculeDatapoint.from_smi(smi) for smi in smis]

#Get molecule dataset
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=featurizers.MultiHotAtomFeaturizer.v1())
test_dset = data.MoleculeDataset(test_data, featurizer=featurizer)
test_loader = data.build_dataloader(test_dset, shuffle=False)

all_preds = make_prediction(models_list, test_loader)

all_preds_array = np.stack(all_preds, axis=1)
columns = [f"pred_{i+1}" for i in range(len(models_list))]
df_test[columns] = all_preds_array
df_test[columns] = 1240/df_test[columns]
df_test['pred_avg'] = np.mean(all_preds_array, axis=1)
df_test['pred_avg'] = 1240/df_test['pred_avg']
df_test

## Predict experimental peak with model trained on Deep4Chem training set

In [ ]:
#Change model input and load model
checkpoint_paths = ['models/lambda_max_abs_v2.1/chemprop/deep4chem/production/fold_0/model_0/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop/deep4chem/production/fold_0/model_1/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop/deep4chem/production/fold_0/model_2/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop/deep4chem/production/fold_0/model_3/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop/deep4chem/production/fold_0/model_4/model.pt']

models_list = []
for path in checkpoint_paths:
  model = models.multi.MulticomponentMPNN.load_from_checkpoint(path, map_location="cpu")
  models_list.append(model)

#Get Smiles
smiles_columns = ["smiles","solvent"]
smiss = df_test[smiles_columns].values
print(smiss[:5])

#Get molecule datapoints
n_components = len(smiles_columns)
test_datapointss = [[data.MoleculeDatapoint.from_smi(smi) for smi in smiss[:, i]] for i in range(n_components)]

#Get molecule dataset
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=featurizers.MultiHotAtomFeaturizer.v1())
test_dsets = [data.MoleculeDataset(test_datapoints, featurizer) for test_datapoints in test_datapointss]
test_mcdset = data.MulticomponentDataset(test_dsets)
test_loader = data.build_dataloader(test_mcdset, shuffle=False)

all_preds = make_prediction(models_list, test_loader)

all_preds_array = np.stack(all_preds, axis=1)
columns = [f"pred_{i+1}" for i in range(len(models_list))]
df_test[columns] = all_preds_array
df_test['pred_avg'] = np.mean(all_preds_array, axis=1)
df_test

## Predict experimental peak with multi-fidelity model

In [ ]:
#Run first prediction
checkpoint_paths = ['models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_0/model.pt',
                    'models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_1/model.pt',
                    'models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_2/model.pt',
                    'models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_3/model.pt',
                    'models/lambda_max_abs_wb97xd3_v2.1/chemprop/all_wb97xd3/production/fold_0/model_4/model.pt']

models_list = []
for path in checkpoint_paths:
  model = models.MPNN.load_from_checkpoint(path, map_location="cpu")
  models_list.append(model)

#Get Smiles
smiles_columns = "smiles"
smis = df_test[smiles_columns]
print(smis)

#Get molecule datapoints
test_data = [data.MoleculeDatapoint.from_smi(smi) for smi in smis]

#Get molecule dataset
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=featurizers.MultiHotAtomFeaturizer.v1())
test_dset = data.MoleculeDataset(test_data, featurizer=featurizer)
test_loader = data.build_dataloader(test_dset, shuffle=False)

all_preds = []

for model in models_list:
    with torch.inference_mode():
        trainer = pl.Trainer(
            logger=None,
            enable_progress_bar=True,
            accelerator="cpu",
            devices=1
        )
        test_preds = trainer.predict(model, test_loader)

    test_preds = np.concatenate(test_preds, axis=0).ravel()
    all_preds.append(test_preds)

all_preds_array = np.stack(all_preds, axis=1)
columns = [f"pred_{i+1}" for i in range(len(models_list))]
df_test[columns] = all_preds_array
avg_preds = np.mean(all_preds_array, axis=1)
df_test['pred_avg'] = avg_preds
df_test

#Convert results into features
test_data_with_xd = [
    data.MoleculeDatapoint.from_smi(smi, x_d=np.array([pred]))
    for smi, pred in zip(smis, avg_preds)
]

test_dset_with_xd = data.MoleculeDataset(test_data_with_xd, featurizer=featurizer)
test_loader_with_xd = data.build_dataloader(test_dset_with_xd, shuffle=False)

#Run second prediction
checkpoint_paths = ['models/lambda_max_abs_v2.1/chemprop_tddft/combined/production/fold_0/model_0/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop_tddft/combined/production/fold_0/model_1/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop_tddft/combined/production/fold_0/model_2/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop_tddft/combined/production/fold_0/model_3/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop_tddft/combined/production/fold_0/model_4/model.pt']

models_list = []
for path in checkpoint_paths:
  model = models.multi.MulticomponentMPNN.load_from_checkpoint(path, map_location="cpu")
  models_list.append(model)

#Get Smiles
smiles_columns = ["smiles","solvent"]
smiss = df_test[smiles_columns].values
print(smiss[:5])

#Get molecule datapoints
n_components = len(smiles_columns)
test_datapointss = [[data.MoleculeDatapoint.from_smi(smi) for smi in smiss[:, i]] for i in range(n_components)]

#Get molecule dataset
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=featurizers.MultiHotAtomFeaturizer.v1())
test_dsets = [data.MoleculeDataset(test_datapoints, featurizer) for test_datapoints in test_datapointss]
test_mcdset = data.MulticomponentDataset(test_dsets)
test_loader = data.build_dataloader(test_mcdset, shuffle=False)

#Show results
make_prediction()

all_preds_array = np.stack(all_preds, axis=1)
columns = [f"pred_{i+1}" for i in range(len(models_list))]
df_test[columns] = all_preds_array
avg_preds = np.mean(all_preds_array, axis=1)
df_test['pred_avg'] = avg_preds
df_test

## Predict experimental peak with model trained on combined training set (with ensemble variance)

In [ ]:
#Change model input and load model
checkpoint_paths = ['models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_0/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_1/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_2/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_3/model.pt',
                    'models/lambda_max_abs_v2.1/chemprop/combined/production/fold_0/model_4/model.pt']

models_list = []
for path in checkpoint_paths:
  model = models.multi.MulticomponentMPNN.load_from_checkpoint(path, map_location="cpu")
  models_list.append(model)

#Get Smiles
smiles_columns = ["smiles","solvent"]
smiss = df_test[smiles_columns].values
print(smiss[:5])

#Get molecule datapoints
n_components = len(smiles_columns)
test_datapointss = [[data.MoleculeDatapoint.from_smi(smi) for smi in smiss[:, i]] for i in range(n_components)]

#Get molecule dataset
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=featurizers.MultiHotAtomFeaturizer.v1())
test_dsets = [data.MoleculeDataset(test_datapoints, featurizer) for test_datapoints in test_datapointss]
test_mcdset = data.MulticomponentDataset(test_dsets)
test_loader = data.build_dataloader(test_mcdset, shuffle=False)

all_preds = make_prediction(models_list, test_loader)

all_preds_array = np.stack(all_preds, axis=1)
columns = [f"pred_{i+1}" for i in range(len(models_list))]
df_test[columns] = all_preds_array
df_test['pred_avg'] = np.mean(all_preds_array, axis=1)
df_test['variance'] = np.var(all_preds_array, axis=1)
df_test